In [18]:
import json
import os
import time
 

In [19]:
import pandas as pd
import requests

In [20]:
BASE = "data"
RAW = os.path.join(BASE, "raw")
CLEAN = os.path.join(BASE, "clean")
OUT_PATH = os.path.join(RAW, "pages_missing.jsonl")
MISS_PATH = os.path.join(RAW, "missing_candidates.jsonl")

In [21]:
API = "https://awoiaf.westeros.org/api.php"
HEADERS = {"User-Agent": "asoiaf-rag-portfolio/0.1 (academic SongOfIceAndFireMind; contact: bryan.beau.186@gmail.com)"}
DELAY = 1.5
MAX_RETRIES = 4

In [22]:
def api_get(params):
    resp = requests.get(API, params={**params, "format": "json"}, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return resp.json()
 

In [23]:

def fetch_page(title):
    params = {
        "action": "query",
        "prop": "extracts|links|categories|info",
        "explaintext": 1,
        "exsectionformat": "wiki",
        "pllimit": "max",
        "plnamespace": 0,
        "cllimit": "max",
        "inprop": "url",
        "titles": title,
        "redirects": 1,
    }
    record = None
    while True:
        data = api_get(params)
        page = next(iter(data["query"]["pages"].values()))
        if "missing" in page or "invalid" in page:
            return None
        if record is None:
            record = {
                "title": page["title"],
                "pageid": page["pageid"],
                "url": page.get("fullurl"),
                "text_wiki": page.get("extract", ""),
                "links": [],
                "categories": [],
            }
        record["links"] += [l["title"] for l in page.get("links", [])]
        record["categories"] += [c["title"] for c in page.get("categories", [])]
        if "continue" not in data:
            record["links"] = sorted(set(record["links"]))
            record["categories"] = sorted(set(record["categories"]))
            return record
        params.update(data["continue"])
        time.sleep(DELAY)


In [24]:
def search_candidates(title):
    data = api_get({
        "action": "query",
        "list": "search",
        "srsearch": title,
        "srnamespace": 0,
        "srlimit": 3,
    })
    return [hit["title"] for hit in data["query"]["search"]]
 
 

In [25]:
def with_retries(fn, arg):
    delay = DELAY
    for attempt in range(MAX_RETRIES):
        try:
            return fn(arg), None
        except Exception as exc:
            if attempt == MAX_RETRIES - 1:
                return None, repr(exc)
            time.sleep(delay)
            delay *= 3
    return None, "unreachable"
 

In [26]:
def read_jsonl(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows

In [27]:
have = set()
for name in ("pages.jsonl", "pages_extra.jsonl"):
    for row in read_jsonl(os.path.join(RAW, name)):
        have.add(row["title"])
        have.add(row["requested_title"])
 

In [28]:
chars = pd.read_json(os.path.join(CLEAN, "characters.jsonl"), lines=True)
battles = pd.read_json(os.path.join(CLEAN, "battles.jsonl"), lines=True)
 

In [29]:
targets = {}
for kind, names in (("character", chars["name"]), ("battle", battles["name"])):
    for name in names:
        if name not in have:
            targets.setdefault(name, kind)

In [30]:
done = {r["requested_title"] for r in read_jsonl(OUT_PATH) + read_jsonl(MISS_PATH)}

In [31]:
pending = [t for t in targets if t not in done]

In [32]:
print(len(targets), len(done), len(pending))

840 840 0


In [16]:
start = time.time()
found = 0
missing = 0
 

In [ ]:
with open(OUT_PATH, "a", encoding="utf-8") as out_file, \
     open(MISS_PATH, "a", encoding="utf-8") as miss_file:
 
    for i, title in enumerate(pending, 1):
        record, error = with_retries(fetch_page, title)
 
        if record is not None:
            record["requested_title"] = title
            record["kind"] = targets[title]
            record["chars"] = len(record["text_wiki"])
            out_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_file.flush()
            found += 1
        else:
            candidates, search_error = [], None
            if error is None:
                time.sleep(DELAY)
                candidates, search_error = with_retries(search_candidates, title)
            miss_file.write(json.dumps({
                "requested_title": title,
                "kind": targets[title],
                "error": error or search_error,
                "candidates": candidates or [],
            }, ensure_ascii=False) + "\n")
            miss_file.flush()
            missing += 1
 
        if i % 50 == 0:
            left = (time.time() - start) / i * (len(pending) - i) / 60
            print(f"{i}/{len(pending)} found={found} missing={missing} eta_min={left:.1f}")
 
        time.sleep(DELAY)

In [ ]:
import json

In [34]:

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


pages = read_jsonl(f"{RAW}/pages.jsonl") + read_jsonl(f"{RAW}/pages_extra.jsonl")

by_title = {}
for p in pages:
    by_title[p["title"]] = p
    by_title[p["requested_title"]] = p

chars = pd.read_json(f"{CLEAN}/characters.jsonl", lines=True)
chars["match"] = chars["name"].isin(by_title)
chars["disambig"] = chars["name"].map(
    lambda n: n in by_title and any("Disambiguation" in c for c in by_title[n]["categories"])
)

print(chars["match"].sum(), len(chars), round(chars["match"].mean(), 3))
print(chars["disambig"].sum())
print(chars.groupby("first_book")["match"].mean().round(2))
print(chars[~chars["match"]]["name"].head(40).tolist())


#%%
bc = pd.read_json(f"{CLEAN}/battles.jsonl", lines=True)
print(bc[bc[["location", "summer", "attacker_commanders"]].isna().any(axis=1)][["name", "year"]])
print(bc["name"].isin(by_title).sum(), len(bc))

99 917 0.108
2
first_book
1    0.15
2    0.08
3    0.09
4    0.08
5    0.15
Name: match, dtype: float64
['Addam Marbrand', 'Aegon Frey (Jinglebell)', 'Adrack Humble', 'Aemon Costayne', 'Aemon Estermont', 'Aenys Frey', 'Aethan', 'Aggar', 'Aggo', 'Alan of Rosby', 'Alayaya', 'Albar Royce', 'Albett', 'Alebelly', 'Alesander Staedmon', 'Alia of Braavos', 'Allard Seaworth', 'Alliser Thorne', 'Alyn', 'Alyn Ambrose', 'Alyn Estermont', 'Alyn Stackspear', 'Alys Karstark', 'Alysane Mormont', 'Alyx Frey', 'Ambrode', 'Amory Lorch', 'Andar Royce', 'Andrew Estermont', 'Andrey Dalt', 'Andrik', 'Anguy', 'Antario Jast', 'Anvil Ryn', 'Anya Waynwood', 'Archibald Yronwood', 'Ardrian Celtigar', 'Areo Hotah', 'Arneld', 'Arnolf Karstark']
                            name  year
22  Battle of the Burning Septry   299
23       Battle of the Ruby Ford   299
15 38


In [35]:
#%%
miss = read_jsonl(f"{RAW}/missing_candidates.jsonl")
for m in miss:
    print(m["kind"], "|", m["requested_title"], "->", m["candidates"], m["error"])


#%%
new = read_jsonl(f"{RAW}/pages_missing.jsonl")
old = read_jsonl(f"{RAW}/pages.jsonl") + read_jsonl(f"{RAW}/pages_extra.jsonl")
old_titles = {p["title"] for p in old}

titles_new = [p["title"] for p in new]
print("rows", len(new), "unique titles", len(set(titles_new)))
print("redirected:", sum(p["title"] != p["requested_title"] for p in new))
print("already in corpus:", len(set(titles_new) & old_titles))
print("short (<200 chars):", sum(len(p["text_wiki"]) < 200 for p in new))

disamb = [p for p in new if any("Disambiguation" in c for c in p["categories"])]
print("disambiguation:", len(disamb), [p["requested_title"] for p in disamb][:25])


#%%
by_title = {}
for p in old + new:
    by_title[p["title"]] = p
    by_title[p["requested_title"]] = p

chars["match"] = chars["name"].isin(by_title)
chars["disambig"] = chars["name"].map(
    lambda n: n in by_title and any("Disambiguation" in c for c in by_title[n]["categories"])
)

print(chars["match"].sum(), len(chars), round(chars["match"].mean(), 3))
print("matched but disambiguation:", int((chars["match"] & chars["disambig"]).sum()))
print("still unmatched:", chars[~chars["match"]]["name"].tolist())

bc["match"] = bc["name"].isin(by_title)
print("battles", bc["match"].sum(), len(bc))

character | Pate (Night's Watch) -> ['Pate (ranger)', "Jaqen H'ghar", 'Samwell Tarly'] None
character | Yezzan Zo Qaggaz -> ['Yezzan zo Qaggaz', 'Yunkai', 'A Dance with Dragons-Chapter 47'] None
character | Torwynd the Tame -> ['Torwynd', 'Munda', 'A Feast for Crows-Appendix'] None
battle | Battle of the Burning Septry -> ['Battle at the burning septry', 'Burning septry', '299 AC'] None
battle | Invasion of Ryamsport, Vinetown, and Starfish Harbor -> ['War of the Five Kings', 'Invasion of the Arbor', 'Arbor'] None
battle | Second Seige of Storm's End -> [] None
rows 834 unique titles 834
redirected: 83
already in corpus: 31
short (<200 chars): 6
disambiguation: 8 ['Byron', 'Elwood', 'Grazdan', 'Lew', 'Lyonel', 'Maris', 'Owen', 'Walton']
914 917 0.997
matched but disambiguation: 10
still unmatched: ["Pate (Night's Watch)", 'Yezzan Zo Qaggaz', 'Torwynd the Tame']
battles 35 38


In [36]:
#%%
import time

import requests

API = "https://awoiaf.westeros.org/api.php"
HEADERS = {"User-Agent": "asoiaf-rag-portfolio/0.1 (academic SongOfIceAndFireMind; contact: bryan.beau.186@gmail.com)"}


def intro(title):
    params = {"action": "query", "prop": "extracts", "explaintext": 1, "exintro": 1,
              "titles": title, "redirects": 1, "format": "json"}
    r = requests.get(API, params=params, headers=HEADERS, timeout=60)
    page = next(iter(r.json()["query"]["pages"].values()))
    time.sleep(1.5)
    return page.get("extract", "MISSING")[:220].replace("\n", " ")


for t in ["Pate (ranger)", "Yezzan zo Qaggaz", "Torwynd", "Battle at the burning septry",
          "Invasion of the Arbor", "Siege of Storm's End"]:
    print(t, "->", intro(t))


#%%
def text_of(p):
    return p.get("text_wiki") or p.get("text") or ""


def allegiance_ok(row):
    p = by_title.get(row["name"])
    if p is None or pd.isna(row["allegiance"]):
        return None
    return row["allegiance"] in " ".join(p["categories"]) or row["allegiance"] in text_of(p)[:1500]


chars["ally_ok"] = chars.apply(allegiance_ok, axis=1)
checked = chars["ally_ok"].dropna().astype(bool)
print(len(checked), round(checked.mean(), 3))
print(chars[chars["ally_ok"] == False][["name", "allegiance"]].head(40).to_string())
print(chars[chars["match"] & chars["disambig"]][["name", "allegiance", "first_book"]].to_string())

Pate (ranger) -> Pate is a ranger in the Night's Watch.
Yezzan zo Qaggaz -> Yezzan zo Qaggaz is an extremely wealthy slave trader and one of the Wise Masters of Yunkai. He is said to be the richest man in the Yellow City, and has great influence because of this. In the television adaptation Game
Torwynd -> Torwynd, known as Torwynd the Tame, is a son of Tormund.
Battle at the burning septry -> The battle at the burning septry occurs in the riverlands during the War of the Five Kings.
Invasion of the Arbor -> The invasion of the Arbor takes place in 300 AC during the second phase of the War of the Five Kings, and is part of the ironborn campaign in the Reach. Following the taking of the Shields, Euron Greyjoy, the new King of
Siege of Storm's End -> The siege of Storm's End took place near the end of Robert's Rebellion. Lord Mace Tyrell and most of his forces from the Reach besieged Storm's End, which was held by Stannis Baratheon, for the better part of a year. The
662 0.932
          

In [37]:
#%%
dis = chars[chars["match"] & chars["disambig"]]
for _, r in dis.iterrows():
    p = by_title[r["name"]]
    print("##", r["name"], "|", r["allegiance"], "| first book", r["first_book"])
    print(text_of(p)[:400].replace("\n", " "))
    print()




## Aegon Targaryen | Targaryen | first book 5
Aegon Targaryen is the name of several members of House Targaryen. These include:  Aegon Targaryen, Lord of Dragonstone before the War of Conquest, eldest son of Daenys and  Gaemon Targaryen Aegon I Targaryen, also known as Aegon the Conqueror and Aegon the Dragon, first Targaryen king Aegon Targaryen, also known as Aegon the Uncrowned, eldest son and heir of King Aenys I Targaryen Aegon Targaryen

## Byron | Arryn | first book 4
Byron is a male name in Westeros. It may refer to:  Ser Byron the Beautiful, a hedge knight Ser Byron Swann, an attempted dragonslayer during the Dance of the Dragons   === See also === Byren Flowers, a supporter of Daemon Blackfyre during the First Blackfyre Rebellion Bryan (disambiguation) Bryen (disambiguation)

## Elwood | Tully | first book 3
Elwood is a common male name in Westeros. It may refer to:  Elwood (guard) Elwood Blackbar Elwood Harte Elwood Meadows

## Grazdan | None | first book 3
The first name Gr

In [38]:
#%%
single = ["Arneld", "Becca", "Dudley", "Kerwin", "Lamprey", "Mallor", "Meliana",
          "Munciter", "Ossy", "Squirrel", "Stiv", "Wallen", "Willum", "Jate"]
for name in single:
    p = by_title[name]
    print(name, "|", p["title"], "|", text_of(p)[:140].replace("\n", " "))

Arneld | Arneld | Arneld is a knight in service at King's Landing. He may be a member of the gold cloaks.   == Recent Events ==   === A Clash of Kings === Dur
Becca | Becca | Becca is a former lover of Robert Baratheon.   == Appearance == Becca has black hair and sweet, large eyes.   == Recent Events ==   === A Ga
Dudley | Dudley | Dudley is a knight in the riverlands.   == Recent Events ==   === A Storm of Swords === Ser Dudley was slain in the riverlands during the Wa
Kerwin | Kerwin | Kerwin is a maester of the Citadel in service to House Chester of Greenshield.   == Appearance ==  Victarion Greyjoy considers Kerwin to be 
Lamprey | Lamprey | Lampreys are jawless fish which can be confused with eels.   == Range == Lampreys have been found in the narrow sea, near White Harbor, in t
Mallor | Mallor | Mallor was a knight during the reign of King Aerys I Targaryen.   == History == Ser Mallor attended the wedding tourney at Whitewalls. His s
Meliana | Meliana | Meliana, known as Lady Mel

In [40]:
import re

In [41]:
for _, r in chars[chars["match"] & chars["disambig"]].iterrows():
    page = by_title[r["name"]]
    related = [l for l in page["links"] if r["name"].lower() in l.lower()]
    print(r["name"], "|", r["allegiance"], "| book", r["first_book"])
    print(related)
    print()


Aegon Targaryen | Targaryen | book 5
['Aegon Targaryen (Young Griff)', 'Aegon Targaryen (son of Aenys I)', 'Aegon Targaryen (son of Aerys II)', 'Aegon Targaryen (son of Baelon)', 'Aegon Targaryen (son of Gaemon)', 'Aegon Targaryen (son of Jaehaerys I)', 'Aegon Targaryen (son of Rhaegar)', 'Maegon Targaryen', 'Vaegon Targaryen']

Byron | Arryn | book 4
['Byron Swann', 'Byron the Beautiful']

Elwood | Tully | book 3
['Elwood (guard)', 'Elwood Blackbar', 'Elwood Harte', 'Elwood Meadows']

Grazdan | None | book 3
['Grazdan (fat one)', 'Grazdan (spiked beard)', 'Grazdan (tall one)', 'Grazdan mo Eraz', 'Grazdan mo Ullhor', 'Grazdan the Great', 'Grazdan zo Galare']

Leo Tyrell | Tyrell | book 4
['Leo Tyrell (Longthorn)', 'Leo Tyrell (son of Gareth)', 'Leo Tyrell (son of Moryn)', 'Leo Tyrell (son of Victor)']

Lew | Stark | book 1
['Left Hand Lew', 'Lew (guard)', 'Lewyn Martell', 'Lewys', 'Lewys Piper', 'Long Lew']

Lyonel | None | book 2
['Lyonel (knight)', 'Lyonel Baratheon', 'Lyonel Bentley

In [43]:
BOOKS = ["A Game of Thrones", "A Clash of Kings", "A Storm of Swords",
         "A Feast for Crows", "A Dance with Dragons"]
FLAGS = ["GoT", "CoK", "SoS", "FfC", "DwD"]


In [44]:
text_by_key = {}
for fname, field in (("pages_wiki", "text_wiki"), ("pages_extra", "text"), ("pages_missing", "text_wiki")):
    for row in read_jsonl(f"{RAW}/{fname}.jsonl"):
        text_by_key[row["title"]] = row[field]
        text_by_key[row["requested_title"]] = row[field]

In [45]:
raw_d = pd.read_csv(f"{RAW}/character-deaths.csv", keep_default_na=False, na_values=[""])


In [46]:
def audit(i):
    page = by_title.get(chars.loc[i, "name"])
    if page is None:
        return "no_page"
    if any("Disambiguation" in c for c in page["categories"]):
        return "disambig"
    if not any("Character" in c for c in page["categories"]):
        return "not_character"
    text = text_by_key.get(page["title"], "")
    in_page = {n for n, b in enumerate(BOOKS, 1) if re.search(rf"^=+ {re.escape(b)} =+$", text, re.M)}
    kaggle = {n for n, col in enumerate(FLAGS, 1) if raw_d.loc[i, col] == 1}
    if not in_page:
        return "no_book_sections"
    return "ok" if in_page & kaggle else "book_conflict"

In [47]:
chars["audit"] = [audit(i) for i in chars.index]
print(chars["audit"].value_counts())
for status in ("not_character", "book_conflict"):
    print(status, chars[chars["audit"] == status]["name"].tolist())

audit
ok                  887
no_book_sections     12
disambig             10
not_character         3
no_page               3
book_conflict         2
Name: count, dtype: int64
not_character ['Elder Brother', 'Lamprey', 'Squirrel']
book_conflict ['Blane', 'Myles']


In [48]:
#%%
print(chars[chars["audit"] == "no_book_sections"][["name", "allegiance", "first_book"]].to_string())
print(chars[chars["name"].isin(["Blane", "Myles"])][["name_key", "allegiance", "first_book", "audit"]])

                  name allegiance  first_book
51               Arson   Wildling           3
78   Bellonara Otherys       None           4
258             Frynne    Martell           4
353             Holger       None           2
405              Jommy  Baratheon           2
417            Joseran    Greyjoy           4
506             Mallor  Lannister           1
535               Meha   Wildling           5
573           Munciter  Lannister           2
655      Qarl Shepherd    Greyjoy           5
682       Raymar Royce      Arryn           3
843               Umar       None           1
            name_key     allegiance  first_book          audit
97             Blane  Night's Watch           2  book_conflict
581  Myles (Martell)        Martell           4             ok
588    Myles (Tully)          Tully           2  book_conflict


In [49]:
fixes = read_jsonl(f"{RAW}/pages_fixes.jsonl")
for row in fixes:
    for key in (row["title"], row["requested_title"]):
        by_title[key] = row
        text_by_key[key] = row["text_wiki"]



In [50]:
def book_set(title):
    text = text_by_key.get(title, "")
    return {n for n, b in enumerate(BOOKS, 1) if re.search(rf"^=+ {re.escape(b)} =+$", text, re.M)}



In [51]:
for i in chars[chars["audit"] == "disambig"].index:
    name, ally = chars.loc[i, "name"], chars.loc[i, "allegiance"]
    ally = None if pd.isna(ally) else ally
    kb = {n for n, col in enumerate(FLAGS, 1) if raw_d.loc[i, col] == 1}
    print("##", name, "| kaggle books", sorted(kb), "| allegiance", ally)
    for link in by_title[name]["links"]:
        if name.lower() not in link.lower() or link not in by_title:
            continue
        page = by_title[link]
        is_char = any("Character" in c for c in page["categories"])
        ally_ok = bool(ally) and ally in " ".join(page["categories"]) + text_by_key.get(link, "")[:1500]
        flag = "MATCH" if is_char and book_set(link) & kb else ""
        print("  ", link, "|", "char" if is_char else "not-char", "| books", sorted(book_set(link)),
              "| ally" if ally_ok else "|", flag)



## Aegon Targaryen | kaggle books [5] | allegiance Targaryen
   Aegon Targaryen (Young Griff) | char | books [5] | ally MATCH
   Aegon Targaryen (son of Aenys I) | char | books [] | ally 
   Aegon Targaryen (son of Aerys II) | char | books [] | ally 
   Aegon Targaryen (son of Baelon) | char | books [] | ally 
   Aegon Targaryen (son of Gaemon) | char | books [] | ally 
   Aegon Targaryen (son of Jaehaerys I) | char | books [] | ally 
   Aegon Targaryen (son of Rhaegar) | char | books [2, 3, 5] | ally MATCH
   Maegon Targaryen | char | books [] | ally 
   Vaegon Targaryen | char | books [] | ally 
## Byron | kaggle books [4] | allegiance Arryn
   Byron Swann | char | books [5] | 
   Byron the Beautiful | char | books [4] | MATCH
## Elwood | kaggle books [3] | allegiance Tully
   Elwood (guard) | char | books [3] | ally MATCH
   Elwood Blackbar | char | books [] | 
   Elwood Harte | char | books [3] | MATCH
   Elwood Meadows | char | books [2, 3, 5] | MATCH
## Grazdan | kaggle books [3]

In [52]:
for t in ["Myles", "Myles Smallwood", "Blane"]:
    p = by_title.get(t)
    print("##", t, "|", "NOT DOWNLOADED" if p is None else p["title"])
    if p is not None:
        text = text_by_key[p["title"]]
        print(re.findall(r"^=+ .+ =+$", text, re.M))
        print(text[:300].replace("\n", " "))

## Myles | Myles
['== Appearance and Character ==', '== Recent Events ==', '=== A Feast for Crows ===', '=== A Dance with Dragons ===', '== References ==']
Myles is a maester in service to House Martell, along with the older Caleotte.   == Appearance and Character == The young Myles has a silky, perfumed beard and a servile smile. Myles wears the grey robes of a maester.   == Recent Events ==   === A Feast for Crows === While Maester Caleotte is with D
## Myles Smallwood | Myles Smallwood
['== Appearance and Character ==', '== History ==', '== Recent Events ==', '=== A Storm of Swords ===', '== Quotes by Myles ==', '== Quotes about Myles ==', '== References ==']
Myles Smallwood was the Lord of Acorn Hall and the head of House Smallwood during the reigns of the kings Aenys I, Maegor I, and Jaehaerys I Targaryen. Renowned for his courage, he briefly served as the Hand of the King to Jaehaerys I.   == Appearance and Character == Myle's face and body bore the s
## Blane | Blane
['== Recent

In [53]:
PICK = {
    "Aegon Targaryen": "Aegon Targaryen (Young Griff)",
    "Byron": "Byron the Beautiful",
    "Elwood": "Elwood (guard)",
    "Leo Tyrell": "Leo Tyrell (son of Moryn)",
    "Lew": "Lew (guard)",
    "Maris": "Black Maris",
    "Owen": "Owen the Oaf",
    "Walton": "Steelshanks Walton",
    "Pate (Night's Watch)": "Pate (ranger)",
    "Yezzan Zo Qaggaz": "Yezzan zo Qaggaz",
    "Torwynd the Tame": "Torwynd",
    "Myles (Martell)": "Myles",
}

In [54]:
NO_PAGE = {"Grazdan", "Lyonel", "Elder Brother", "Lamprey", "Squirrel", "Mallor", "Myles (Tully)"}
BATTLE_PICK = {"Battle of the Burning Septry": "Battle at the burning septry"}


In [55]:
def character_page(row):
    key = row["name_key"]
    if key in NO_PAGE:
        return None
    if key in PICK:
        return by_title[PICK[key]]["title"]
    page = by_title.get(row["name"])
    return page["title"] if page else None


In [56]:
def battle_page(name):
    key = BATTLE_PICK.get(name, name)
    return by_title[key]["title"] if key in by_title else None


In [57]:
chars["page_title"] = chars.apply(character_page, axis=1)
bc["page_title"] = bc["name"].map(battle_page)
print(chars["page_title"].notna().sum(), len(chars), bc["page_title"].notna().sum(), len(bc))


910 917 36 38


In [58]:
cols = ["name_key", "name", "allegiance", "male", "noble", "first_book", "intro_chapter",
        "is_dead", "death_year", "death_book", "death_chapter", "page_title"]
chars[cols].to_json(f"{CLEAN}/characters_linked.jsonl", orient="records", lines=True, force_ascii=False)
bc.drop(columns=["match"], errors="ignore").to_json(
    f"{CLEAN}/battles_linked.jsonl", orient="records", lines=True, force_ascii=False)


In [59]:
wiki_text = {r["requested_title"]: r["text_wiki"] for r in read_jsonl(f"{RAW}/pages_wiki.jsonl")}
corpus = {}

In [60]:
def add(rec, text, source):
    title = rec["title"]
    kinds = rec.get("kinds") or ([rec["kind"]] if rec.get("kind") else [])
    if title in corpus:
        corpus[title]["kinds"] = sorted(set(corpus[title]["kinds"]) | set(kinds))
        return
    corpus[title] = {
        "title": title, "pageid": rec["pageid"], "url": rec["url"], "text": text,
        "links": rec["links"], "categories": rec["categories"],
        "kinds": sorted(set(kinds)), "source": source,
    }


In [61]:
for rec in read_jsonl(f"{RAW}/pages.jsonl"):
    add(rec, wiki_text[rec["requested_title"]], "base")
for rec in read_jsonl(f"{RAW}/pages_extra.jsonl"):
    add(rec, rec["text"], "extra")
for name in ("pages_missing", "pages_fixes", "pages_expansion"):
    for rec in read_jsonl(f"{RAW}/{name}.jsonl"):
        add(rec, rec["text_wiki"], name.split("_")[1])


In [62]:
with open(f"{CLEAN}/corpus.jsonl", "w", encoding="utf-8") as f:
    for rec in corpus.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [63]:
print(len(corpus), sum(len(r["text"]) for r in corpus.values()))
print(pd.Series([r["source"] for r in corpus.values()]).value_counts())
print([t for t in chars["page_title"].dropna().unique() if t not in corpus])
print([t for t in bc["page_title"].dropna() if t not in corpus])

3539 12124164
base       2248
missing     803
extra       453
fixes        35
Name: count, dtype: int64
[]
[]


In [66]:
from collections import Counter, defaultdict

In [67]:
corpus = {r["title"]: r for r in read_jsonl(f"{CLEAN}/corpus.jsonl")}
HEAD = re.compile(r"^(=+) (.+?) =+$", re.M)

In [68]:
def sections(text):
    marks = list(HEAD.finditer(text))
    out = [("", 0, text[: marks[0].start()] if marks else text)]
    for k, m in enumerate(marks):
        end = marks[k + 1].start() if k + 1 < len(marks) else len(text)
        out.append((m.group(2), len(m.group(1)), text[m.end():end]))
    return out


In [69]:
freq = Counter()
size = defaultdict(int)

In [70]:
for rec in corpus.values():
    for title, level, body in sections(rec["text"]):
        freq[title] += 1
        size[title] += len(body)

In [71]:
total = sum(size.values())
print("intro share:", round(size[""] / total, 3))
for title, n in freq.most_common(35):
    if title:
        print(n, round(size[title] / total, 3), title)

intro share: 0.063
3200 0.001 References
1921 0.0 Recent Events
1761 0.089 History
1064 0.062 A Storm of Swords
1044 0.051 A Feast for Crows
973 0.0 Quotes
920 0.055 A Clash of Kings
919 0.054 A Dance with Dragons
779 0.002 Family
641 0.043 A Game of Thrones
568 0.028 Appearance and Character
483 0.02 Birth
457 0.0 Notes
354 0.118 Synopsis
271 0.002 External links
262 0.005 Behind the Scenes
252 0.0 Mentions
229 0.007 Historical Members
219 0.004 Appearance
202 0.003 References and Notes
200 0.006 The Winds of Winter
194 0.0 Character List
169 0.013 Aftermath
149 0.008 Layout
149 0.001 Character
145 0.014 Prelude
141 0.006 Death
129 0.0 Character list
122 0.001 Births
112 0.001 See also
100 0.005 Culture
98 0.001 Events
81 0.005 Battle
73 0.0 External Links


In [72]:
junk = sum(v for t, v in size.items()
           if t.startswith(("References", "External links", "Quotes by", "Quotes about", "Gallery")))
print("candidate junk share:", round(junk / total, 3))



candidate junk share: 0.007


In [73]:
print("book subsection share:", round(sum(size[b] for b in BOOKS) / total, 3))
print("pages with book subsections:",
      sum(any(t in BOOKS for t, _, _ in sections(r["text"])) for r in corpus.values()))


book subsection share: 0.265
pages with book subsections: 1952


In [74]:
chap = re.compile(r"^(" + "|".join(map(re.escape, BOOKS)) + r")-(.+)$")
chapters = [t for t in corpus if chap.match(t)]

In [75]:
print("chapter-like pages:", len(chapters))
print(Counter(chap.match(t).group(1) for t in chapters))
print([t for t in chapters if not re.search(r"Chapter \d+$", t)][:20])

chapter-like pages: 344
Counter({'A Storm of Swords': 82, 'A Dance with Dragons': 73, 'A Game of Thrones': 73, 'A Clash of Kings': 70, 'A Feast for Crows': 46})
['A Clash of Kings-Prologue', 'A Dance with Dragons-Epilogue', 'A Dance with Dragons-Prologue', 'A Feast for Crows-Prologue', 'A Game of Thrones-Prologue', 'A Storm of Swords-Epilogue', 'A Storm of Swords-Prologue']


In [76]:
length = pd.Series({t: len(r["text"]) for t, r in corpus.items()})

In [77]:
dis = [t for t, r in corpus.items() if any("Disambiguation" in c for c in r["categories"])]
print("disambiguation:", len(dis), "| under 500 chars:", int((length < 500).sum()))
print(Counter(k for r in corpus.values() for k in r["kinds"]).most_common())

disambiguation: 41 | under 500 chars: 645
[('character', 1734), ('house', 511), ('chapter', 356), ('place', 348), ('culture', 181), ('battle', 160), ('event', 103), ('hub', 103), ('religion', 99), ('dragon', 47), ('fix', 35)]


In [78]:
DROP = ("references", "external links", "quotes", "gallery", "see also", "notes", "behind the scenes")
TARGET = 1500
LIMIT = 2200
BOOK_NUM = {b: n for n, b in enumerate(BOOKS, 1)}


In [79]:
linked = pd.read_json(f"{CLEAN}/characters_linked.jsonl", lines=True)
linked = linked[linked["page_title"].notna()]
first_book = {t: int(v) for t, v in linked.groupby("page_title")["first_book"].min().items() if pd.notna(v)}


In [80]:
def is_dropped(path):
    return any(p.lower().startswith(DROP) for p in path)



In [81]:
def parse_sections(text):
    marks = list(HEAD.finditer(text))
    out = [([], text[: marks[0].start()] if marks else text)]
    stack = []
    for k, m in enumerate(marks):
        stack = stack[: len(m.group(1)) - 2] + [m.group(2)]
        end = marks[k + 1].start() if k + 1 < len(marks) else len(text)
        out.append((list(stack), text[m.end():end]))
    return out


In [82]:
def spoiler_for(title, path):
    m = chap.match(title)
    if m:
        return BOOK_NUM[m.group(1)]
    if title.startswith("The Winds of Winter-"):
        return 6
    for part in path:
        if part in BOOK_NUM:
            return BOOK_NUM[part]
    return first_book.get(title, 1)


In [83]:
def pieces(body):
    units = []
    for p in (p.strip() for p in re.split(r"\n+", body)):
        if not p:
            continue
        if len(p) <= LIMIT:
            units.append(p)
        else:
            units += [s for s in re.split(r"(?<=[.!?]) +", p) if s]
    out, cur = [], ""
    for u in units:
        if cur and len(cur) + len(u) + 1 > TARGET:
            out.append(cur)
            cur = u
        else:
            cur = f"{cur}\n{u}" if cur else u
    if cur:
        out.append(cur)
    return out

In [84]:
chunks = []
skipped = Counter()

In [85]:

for title, rec in corpus.items():
    if any("Disambiguation" in c for c in rec["categories"]):
        skipped["disambiguation_pages"] += 1
        continue

    page_chunks = []
    for path, body in parse_sections(rec["text"]):
        if is_dropped(path):
            skipped["dropped_chars"] += len(body)
            continue
        sb = spoiler_for(title, path)
        for text in pieces(body):
            page_chunks.append({"path": path, "text": text, "spoiler_book": sb})

    merged = []
    for c in page_chunks:
        prev = merged[-1] if merged else None
        if (prev and prev["path"] and c["path"]
                and prev["path"][:-1] == c["path"][:-1]
                and prev["spoiler_book"] == c["spoiler_book"]
                and len(prev["text"]) < 600
                and len(prev["text"]) + len(c["text"]) < TARGET):
            prev["text"] += f"\n{c['path'][-1]}: {c['text']}"
        else:
            merged.append(c)

    for n, c in enumerate(merged):
        chunks.append({
            "chunk_id": f"{rec['pageid']}-{n}",
            "page_title": title,
            "url": rec["url"],
            "path": " > ".join([title] + c["path"]),
            "text": c["text"],
            "spoiler_book": c["spoiler_book"],
            "kinds": rec["kinds"],
            "chars": len(c["text"]),
        })

In [86]:
with open(f"{CLEAN}/chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")


In [87]:
ch = pd.DataFrame(chunks)
print(len(ch), int(ch["chars"].sum()), dict(skipped))
print(ch["chars"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).round(0))
print(ch["spoiler_book"].value_counts().sort_index())
print(ch[ch["page_title"] == "Daenerys Targaryen"][["path", "spoiler_book", "chars"]].to_string())


18305 11375043 {'dropped_chars': 174412, 'disambiguation_pages': 41}
count    18305.0
mean       621.0
std        459.0
min          3.0
10%        101.0
50%        515.0
90%       1321.0
99%       1489.0
max       2149.0
Name: chars, dtype: float64
spoiler_book
1    11451
2     1947
3     1831
4     1567
5     1509
Name: count, dtype: int64
                                                           path  spoiler_book  chars
1935                                         Daenerys Targaryen             1    572
1936              Daenerys Targaryen > Appearance and Character             1    979
1937              Daenerys Targaryen > Appearance and Character             1   1297
1938              Daenerys Targaryen > Appearance and Character             1    502
1939                               Daenerys Targaryen > History             1   1117
1940                               Daenerys Targaryen > History             1    925
1941                               Daenerys Targaryen > Histo

In [88]:
print([t for t, r in corpus.items() if "chapter" in r["kinds"] and not chap.match(t)])

['Alayne I (The Winds of Winter)', 'Arianne I (The Winds of Winter)', 'Arianne II (The Winds of Winter)', 'Barristan I (The Winds of Winter)', 'Barristan II (The Winds of Winter)', 'Chapters', 'Chapters Table of contents', 'Mercy (The Winds of Winter)', 'The Forsaken (The Winds of Winter)', 'Theon I (The Winds of Winter)', 'Tyrion II (The Winds of Winter)', 'Victarion I (The Winds of Winter)']


In [89]:
for b in BOOKS:
    nums = sorted(int(m.group(1)) for t in corpus
                  if t.startswith(b + "-") and (m := re.search(r"Chapter (\d+)$", t)))
    print(b, len(nums), "missing", sorted(set(range(1, nums[-1] + 1)) - set(nums)))

A Game of Thrones 72 missing []
A Clash of Kings 69 missing []
A Storm of Swords 80 missing []
A Feast for Crows 45 missing []
A Dance with Dragons 71 missing []


In [90]:
SKIP_PAGES = {"Chapters", "Chapters Table of contents"}

In [91]:

def spoiler_label(title, path):
    m = chap.match(title)
    if m:
        return BOOK_NUM[m.group(1)], "chapter"
    if title.endswith("(The Winds of Winter)"):
        return 6, "chapter"
    for part in path:
        if part in BOOK_NUM:
            return BOOK_NUM[part], "book_section"
        if part == "The Winds of Winter":
            return 6, "book_section"
    return None, "unlabeled"


In [92]:
chunks = []
skipped = Counter()

In [93]:

for title, rec in corpus.items():
    if title in SKIP_PAGES:
        skipped["index_pages"] += 1
        continue
    if any("Disambiguation" in c for c in rec["categories"]):
        skipped["disambiguation_pages"] += 1
        continue

    page_chunks = []
    for path, body in parse_sections(rec["text"]):
        if is_dropped(path):
            skipped["dropped_chars"] += len(body)
            continue
        label, source = spoiler_label(title, path)
        for text in pieces(body):
            page_chunks.append({"path": path, "text": text, "spoiler_book": label, "label_source": source})

    merged = []
    for c in page_chunks:
        if merged:
            prev = merged[-1]
            same = prev["spoiler_book"] == c["spoiler_book"] and prev["label_source"] == c["label_source"]
            sibling = bool(prev["path"]) and bool(c["path"]) and prev["path"][:-1] == c["path"][:-1]
            old_rule = sibling and len(prev["text"]) < 600
            tiny = len(c["text"]) < 150
            if same and (old_rule or tiny) and len(prev["text"]) + len(c["text"]) < TARGET:
                tag = c["path"][-1] if c["path"] else ""
                prev["text"] += f"\n{tag}: {c['text']}" if tag else f"\n{c['text']}"
                continue
        merged.append(c)

    merged = [c for c in merged if len(c["text"]) >= 20]

    for n, c in enumerate(merged):
        chunks.append({
            "chunk_id": f"{rec['pageid']}-{n}",
            "page_title": title,
            "url": rec["url"],
            "path": " > ".join([title] + c["path"]),
            "text": c["text"],
            "spoiler_book": c["spoiler_book"],
            "label_source": c["label_source"],
            "debut_book": first_book.get(title),
            "kinds": rec["kinds"],
            "chars": len(c["text"]),
        })


In [96]:
with open(f"{CLEAN}/chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")



In [97]:
ch = pd.DataFrame(chunks)

In [98]:
print(len(ch), int(ch["chars"].sum()), dict(skipped))
print(ch["chars"].describe(percentiles=[0.1, 0.5, 0.9]).round(0))
print("under 150 chars:", int((ch["chars"] < 150).sum()))
print(ch["label_source"].value_counts())
print(ch.groupby("label_source")["spoiler_book"].value_counts(dropna=False).sort_index())
print(ch[ch["page_title"].str.endswith("(The Winds of Winter)")]["spoiler_book"].value_counts())

17527 11387007 {'dropped_chars': 174412, 'disambiguation_pages': 41, 'index_pages': 2}
count    17527.0
mean       650.0
std        452.0
min         20.0
10%        121.0
50%        559.0
90%       1328.0
max       2149.0
Name: chars, dtype: float64
under 150 chars: 2410
label_source
unlabeled       10547
book_section     5531
chapter          1449
Name: count, dtype: int64
label_source  spoiler_book
book_section  1.0               789
              2.0              1068
              3.0              1242
              4.0              1162
              5.0              1069
              6.0               201
chapter       1.0               453
              2.0               453
              3.0               193
              4.0               139
              5.0               185
              6.0                26
unlabeled     NaN             10547
Name: count, dtype: int64
spoiler_book
6.0    26
Name: count, dtype: int64


In [99]:
print(ch[ch["label_source"] == "chapter"].groupby("spoiler_book")["chars"].agg(["count", "sum"]))


              count     sum
spoiler_book               
1.0             453  445644
2.0             453  448105
3.0             193  175303
4.0             139  140372
5.0             185  186652
6.0              26   21329


In [100]:
pages_ok = {t for t, r in corpus.items() if not any("Disambiguation" in c for c in r["categories"])}
indeg = Counter()
edges = chapter_edges = 0
for t in pages_ok:
    r = corpus[t]
    for l in r["links"]:
        if l in pages_ok and l != t:
            indeg[l] += 1
            edges += 1
            chapter_edges += "chapter" in r["kinds"]


In [102]:
print(len(pages_ok), edges, chapter_edges, round(edges / sum(len(corpus[t]["links"]) for t in pages_ok), 3))
print(indeg.most_common(15))
print("pages with no incoming link:", sum(1 for t in pages_ok if indeg[t] == 0))

3498 216733 28670 0.707
[('A Dance with Dragons', 1824), ('A Feast for Crows', 1634), ('A Storm of Swords', 1570), ('A Game of Thrones', 1367), ('The World of Ice & Fire', 1310), ('A Clash of Kings', 1293), ("King's Landing", 1204), ('Fire & Blood', 917), ('Knight', 886), ('Tyrion Lannister', 861), ("Years after Aegon's Conquest", 849), ('Westeros', 842), ('House Lannister', 823), ('House Targaryen', 817), ('House Stark', 748)]
pages with no incoming link: 20


In [103]:
## Buildup Graph in memory

In [104]:
import heapq
import math

In [105]:
META = set(BOOKS) | {"The World of Ice & Fire", "A World of Ice and Fire",
                     "Fire & Blood", "A Song of Ice and Fire"}
BASE_COST = {"structured": 0.5, "chapter": 1.0, "link": 2.0}
LAM = 0.5

In [106]:
edges = {}

In [107]:

def put(a, b, kind):
    if a == b or a in META or b in META:
        return
    for x, y in ((a, b), (b, a)):
        cur = edges.get((x, y))
        if cur is None or BASE_COST[kind] < BASE_COST[cur]:
            edges[(x, y)] = kind


In [108]:
def is_chapter(title):
    return bool(chap.match(title)) or title.endswith("(The Winds of Winter)")


In [109]:
def resolve(*names):
    for n in names:
        if n in pages_ok:
            return n
    return None


In [110]:
for t in pages_ok:
    for l in corpus[t]["links"]:
        if l in pages_ok:
            put(t, l, "chapter" if is_chapter(t) or is_chapter(l) else "link")


In [111]:
cl = pd.read_json(f"{CLEAN}/characters_linked.jsonl", lines=True)
for _, r in cl[cl["page_title"].notna() & cl["allegiance"].notna()].iterrows():
    house = resolve(f"House {r['allegiance']}", r["allegiance"], f"{r['allegiance']}s")
    if house:
        put(r["page_title"], house, "structured")


In [112]:
def king_titles(value):
    if not isinstance(value, str):
        return []
    parts = [p.strip() for p in value.split("/")]
    if len(parts) == 2:
        parts[0] = f"{parts[0]} {parts[1].split()[-1]}"
    return parts


In [114]:
bl = pd.read_json(f"{CLEAN}/battles_linked.jsonl", lines=True)


In [115]:
for _, r in bl[bl["page_title"].notna()].iterrows():
    houses = [resolve(f"House {h}", h) for h in list(r["attackers"]) + list(r["defenders"])]
    people = (list(r["attacker_commanders"]) + list(r["defender_commanders"])
              + king_titles(r["attacker_king"]) + king_titles(r["defender_king"]))
    place = [r["location"]] if isinstance(r["location"], str) else []
    for target in houses + [resolve(p) for p in people + place]:
        if target:
            put(r["page_title"], target, "structured")

In [116]:
deg = Counter(b for (_, b) in edges)
adj = defaultdict(list)
for (a, b), kind in edges.items():
    adj[a].append((b, BASE_COST[kind] + LAM * math.log1p(deg[b]), kind))



In [117]:
print(len({a for a, _ in edges}), len(edges) // 2)
print({k: v // 2 for k, v in Counter(edges.values()).items()})


3488 138721
{'chapter': 33816, 'link': 103982, 'structured': 923}


In [118]:
def bfs(seed, depth):
    seen = {seed: 0}
    frontier = [seed]
    for d in range(1, depth + 1):
        nxt = []
        for u in frontier:
            for v, _, _ in adj[u]:
                if v not in seen:
                    seen[v] = d
                    nxt.append(v)
        frontier = nxt
    return seen

In [119]:
def dijkstra(seed, budget):
    dist = {seed: 0.0}
    heap = [(0.0, seed)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w, _ in adj[u]:
            nd = d + w
            if nd <= budget and nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist


In [120]:
def shortest_path(src, dst):
    dist, prev, heap = {src: 0.0}, {}, [(0.0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if u == dst:
            break
        if d > dist[u]:
            continue
        for v, w, _ in adj[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v], prev[v] = nd, u
                heapq.heappush(heap, (nd, v))
    if dst not in dist:
        return None
    node, path = dst, [dst]
    while node != src:
        node = prev[node]
        path.append(node)
    return round(dist[dst], 2), path[::-1]



In [121]:
for seed in ["Red Wedding", "Robb Stark", "Meereen"]:
    two_hops = bfs(seed, 2)
    reach = dijkstra(seed, 6.0)
    print(seed, "| bfs depth 2:", len(two_hops), "| dijkstra budget 6:", len(reach))
    print([(t, round(c, 2)) for t, c in sorted(reach.items(), key=lambda kv: kv[1])[1:13]])


Red Wedding | bfs depth 2: 3257 | dijkstra budget 6: 561
[('Walder Rivers', 2.46), ('A Storm of Swords-Chapter 52', 2.73), ('A Storm of Swords-Chapter 47', 3.07), ('A Storm of Swords-Chapter 50', 3.09), ('A Storm of Swords-Chapter 58', 3.09), ('Craven', 3.1), ('A Clash of Kings-Chapter 48', 3.1), ('A Storm of Swords-Chapter 61', 3.1), ('A Storm of Swords-Chapter 65', 3.19), ('A Dance with Dragons-Chapter 19', 3.23), ('A Dance with Dragons-Chapter 51', 3.24), ('Owen Norrey', 3.24)]
Robb Stark | bfs depth 2: 3436 | dijkstra budget 6: 1212
[('A Dance with Dragons-Chapter 40', 2.67), ('A Storm of Swords-Chapter 52', 2.73), ('A Game of Thrones-Chapter 17', 2.76), ('A Game of Thrones-Chapter 10', 2.82), ('Sack of Darry', 2.85), ('Siege of Darry', 2.85), ("Fight at Torrhen's Square", 2.85), ('Siege of Seagard', 2.86), ('Battle under the walls of Riverrun', 2.9), ('A Clash of Kings-Chapter 68', 2.9), ('Fighting at the fords of the Trident', 2.91), ('Battle in the hills below the Golden Tooth',

In [122]:
print(shortest_path("Arya Stark", "Red Wedding"))
print(shortest_path("Daenerys Targaryen", "Meereen"))

(5.01, ['Arya Stark', 'Red Wedding'])
(4.87, ['Daenerys Targaryen', 'Meereen'])


In [124]:
# Dev Notes 
"""
El grafo es muy denso: 138 721 aristas entre 3488 nodos son unos 80 vecinos por nodo (antes conté 216 733
porque contaba cada enlace por dirección). Un BFS a profundidad 2 desde la Boda Roja toca 3257 de 3488 nodos, 
el 93% del grafo, así que sin pesos no sirve para nada. Con costos y presupuesto 6, toca 561, 1212 y 413 nodos.
Ese contraste es tu primera ablación. De paso, el presupuesto es la medida equivocada: para recuperar queremos
los k nodos más cercanos,
no “todo lo que cabe en 6”
"""

'\nEl grafo es muy denso: 138 721 aristas entre 3488 nodos son unos 80 vecinos por nodo (antes conté 216 733\nporque contaba cada enlace por dirección). Un BFS a profundidad 2 desde la Boda Roja toca 3257 de 3488 nodos, \nel 93% del grafo, así que sin pesos no sirve para nada. Con costos y presupuesto 6, toca 561, 1212 y 413 nodos.\nEse contraste es tu primera ablación. De paso, el presupuesto es la medida equivocada: para recuperar queremos\nlos k nodos más cercanos,\nno “todo lo que cabe en 6”\n'

In [125]:
out_links = {t: {l for l in corpus[t]["links"] if l in pages_ok and l != t} for t in pages_ok}



In [126]:
def base_cost(a, b, kind):
    if kind == "link":
        return 1.5 if (b in out_links[a] and a in out_links[b]) else 2.5
    return BASE_COST[kind]


In [127]:
deg = Counter(b for (_, b) in edges)
toll = {t: LAM * math.log1p(deg[t]) for t in deg}
adj = defaultdict(list)

In [128]:
for (a, b), kind in edges.items():
    adj[a].append((b, base_cost(a, b, kind), kind))


In [129]:
kinds = Counter(k for (a, b), k in edges.items() if a < b)
mutual = sum(1 for (a, b), k in edges.items() if a < b and k == "link" and b in out_links[a] and a in out_links[b])


In [130]:
print(dict(kinds), "mutual links:", mutual)

{'chapter': 33816, 'link': 103982, 'structured': 923} mutual links: 50667


In [131]:
def nearest(seed, k):
    dist, seen, done, heap = {seed: 0.0}, set(), [], [(0.0, seed)]
    while heap and len(done) <= k:
        d, u = heapq.heappop(heap)
        if u in seen:
            continue
        seen.add(u)
        done.append((u, d))
        step = 0.0 if u == seed else toll.get(u, 0.0)
        for v, w, _ in adj[u]:
            nd = d + step + w
            if v not in seen and nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return done[1:]

In [132]:
def shortest_path(src, dst):
    dist, prev, seen, heap = {src: 0.0}, {}, set(), [(0.0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if u in seen:
            continue
        seen.add(u)
        if u == dst:
            break
        step = 0.0 if u == src else toll.get(u, 0.0)
        for v, w, _ in adj[u]:
            nd = d + step + w
            if nd < dist.get(v, float("inf")):
                dist[v], prev[v] = nd, u
                heapq.heappush(heap, (nd, v))
    if dst not in dist:
        return None
    node, path = dst, [dst]
    while node != src:
        node = prev[node]
        path.append(node)
    return round(dist[dst], 2), path[::-1]


In [133]:
for seed in ["Red Wedding", "Robb Stark", "Meereen"]:
    print(seed, [(t, round(c, 2)) for t, c in nearest(seed, 20)])

Red Wedding [('House Bolton', 0.5), ('House Frey', 0.5), ('House Stark', 0.5), ('Joffrey Baratheon', 0.5), ('Robb Stark', 0.5), ('Roose Bolton', 0.5), ('Tommen Baratheon', 0.5), ('Walder Frey', 0.5), ('Walder Rivers', 0.5), ('A Clash of Kings-Chapter 45', 1.0), ('A Clash of Kings-Chapter 48', 1.0), ('A Clash of Kings-Chapter 64', 1.0), ('A Clash of Kings-Chapter 66', 1.0), ('A Dance with Dragons-Chapter 19', 1.0), ('A Dance with Dragons-Chapter 29', 1.0), ('A Dance with Dragons-Chapter 37', 1.0), ('A Dance with Dragons-Chapter 46', 1.0), ('A Dance with Dragons-Chapter 48', 1.0), ('A Dance with Dragons-Chapter 51', 1.0), ('A Feast for Crows-Chapter 10', 1.0)]
Robb Stark [('Battle at Duskendale', 0.5), ("Battle at the Mummer's Ford", 0.5), ('Battle in the Whispering Wood', 0.5), ('Battle in the hills below the Golden Tooth', 0.5), ('Battle of Oxcross', 0.5), ('Battle of the Camps', 0.5), ('Battle of the Fords', 0.5), ('Battle on the Green Fork', 0.5), ('Battle outside the gates of Winter

In [134]:
rank = {t: i + 1 for i, (t, _) in enumerate(nearest("Red Wedding", 300))}
for t in ["Robb Stark", "Catelyn Stark", "Walder Frey", "Roose Bolton", "Tywin Lannister", "Edmure Tully"]:
    print(t, rank.get(t))

Robb Stark 5
Catelyn Stark 98
Walder Frey 8
Roose Bolton 6
Tywin Lannister 250
Edmure Tully 107


In [135]:
for a, b in [("Arya Stark", "Red Wedding"), ("Daenerys Targaryen", "Meereen"),
             ("Arya Stark", "Walder Frey"), ("Sansa Stark", "Tywin Lannister")]:
    print(a, "->", b, shortest_path(a, b))

Arya Stark -> Red Wedding (1.5, ['Arya Stark', 'Red Wedding'])
Daenerys Targaryen -> Meereen (1.5, ['Daenerys Targaryen', 'Meereen'])
Arya Stark -> Walder Frey (1.5, ['Arya Stark', 'Walder Frey'])
Sansa Stark -> Tywin Lannister (1.5, ['Sansa Stark', 'Tywin Lannister'])


In [136]:
"""
Los dos arreglos funcionaron en lo que medían. Robb (puesto 5), Roose Bolton (6) y Walder Frey (8) ya están arriba. Pero fíjate en por qué: los 9 vecinos más cercanos de la Boda Roja cuestan exactamente 0.5, es decir, son aristas estructuradas del CSV. Son solo 923 de 138 721 aristas (0.7%), pero salen de las 38 batallas y dominan el top de cada lista. Ese resultado va al README: unos pocos datos limpios pesan más que miles de enlaces ruidosos
"""

'\nLos dos arreglos funcionaron en lo que medían. Robb (puesto 5), Roose Bolton (6) y Walder Frey (8) ya están arriba. Pero fíjate en por qué: los 9 vecinos más cercanos de la Boda Roja cuestan exactamente 0.5, es decir, son aristas estructuradas del CSV. Son solo 923 de 138 721 aristas (0.7%), pero salen de las 38 batallas y dominan el top de cada lista. Ese resultado va al README: unos pocos datos limpios pesan más que miles de enlaces ruidosos\n'

In [137]:
import numpy as np

In [138]:
nodes = sorted({a for a, _ in edges})
idx = {t: i for i, t in enumerate(nodes)}
n = len(nodes)
src = np.array([idx[a] for (a, b) in edges])
dst = np.array([idx[b] for (a, b) in edges])
w = 1.0 / np.array([base_cost(a, b, k) for (a, b), k in edges.items()])
p = w / np.bincount(src, weights=w, minlength=n)[src]

In [139]:
def pagerank(restart, alpha=0.15, iters=50):
    r = restart.copy()
    for _ in range(iters):
        r = alpha * restart + (1 - alpha) * np.bincount(dst, weights=r[src] * p, minlength=n)
    return r

In [141]:
def ppr(seeds):
    restart = np.zeros(n)
    for t, wt in seeds.items():
        restart[idx[t]] = wt
    return pagerank(restart / restart.sum())

In [142]:
glob = pagerank(np.full(n, 1.0 / n))

In [143]:
test = ["Robb Stark", "Catelyn Stark", "Walder Frey", "Roose Bolton", "Tywin Lannister", "Edmure Tully"]
entities = [t for t in nodes if not is_chapter(t) and t != "Red Wedding"]


In [144]:
order = [t for t, _ in nearest("Red Wedding", n) if not is_chapter(t)]
rk = {t: i + 1 for i, t in enumerate(order)}
print("dijkstra", {t: rk.get(t) for t in test}, order[:12])


dijkstra {'Robb Stark': 5, 'Catelyn Stark': 46, 'Walder Frey': 8, 'Roose Bolton': 6, 'Tywin Lannister': 198, 'Edmure Tully': 55} ['House Bolton', 'House Frey', 'House Stark', 'Joffrey Baratheon', 'Robb Stark', 'Roose Bolton', 'Tommen Baratheon', 'Walder Frey', 'Walder Rivers', '299 AC', 'Aegon Frey (son of Stevron)', 'Aenys Frey']


In [145]:
raw = ppr({"Red Wedding": 1.0})
for name, s in {"raw": raw, "sqrt": raw / np.sqrt(glob), "lift": raw / glob}.items():
    order = sorted(entities, key=lambda t: -s[idx[t]])
    rk = {t: i + 1 for i, t in enumerate(order)}
    print(name, {t: rk[t] for t in test}, order[:12])

raw {'Robb Stark': 3, 'Catelyn Stark': 13, 'Walder Frey': 15, 'Roose Bolton': 16, 'Tywin Lannister': 12, 'Edmure Tully': 34} ['House Stark', 'House Lannister', 'Robb Stark', 'Joffrey Baratheon', 'House Frey', 'War of the Five Kings', "King's Landing", 'Jaime Lannister', 'Tommen Baratheon', 'Knight', 'House Bolton', 'Tywin Lannister']
sqrt {'Robb Stark': 3, 'Catelyn Stark': 15, 'Walder Frey': 2, 'Roose Bolton': 6, 'Tywin Lannister': 35, 'Edmure Tully': 13} ['Walder Rivers', 'Walder Frey', 'Robb Stark', 'House Frey', 'House Stark', 'Roose Bolton', 'Joffrey Baratheon', 'House Bolton', 'Tommen Baratheon', 'War of the Five Kings', 'House Lannister', 'Twins']
lift {'Robb Stark': 228, 'Catelyn Stark': 281, 'Walder Frey': 48, 'Roose Bolton': 133, 'Tywin Lannister': 329, 'Edmure Tully': 190} ['Walder Rivers', 'Owen Norrey', 'Garse Goodbrook', 'Tytos Frey', 'Whalen Frey', 'Benfrey Frey', 'Alyx Frey', 'Joyeuse Erenford', 'Olyvar Frey', 'Raymund Frey', 'Wedding of Jaehaerys I Targaryen and Alysann

In [146]:
chapters = sorted([t for t in nodes if is_chapter(t)], key=lambda t: -(raw / np.sqrt(glob))[idx[t]])
print(chapters[:10])


['A Storm of Swords-Chapter 51', 'A Storm of Swords-Chapter 49', 'A Storm of Swords-Epilogue', 'A Storm of Swords-Chapter 14', 'A Feast for Crows-Chapter 44', 'A Storm of Swords-Chapter 45', 'A Game of Thrones-Chapter 59', 'A Feast for Crows-Chapter 38', 'A Game of Thrones-Chapter 63', 'A Storm of Swords-Chapter 35']


In [147]:
for a, b in [("Daenerys Targaryen", "Red Wedding"), ("Bran Stark", "Walder Frey"),
             ("Jon Snow", "Edmure Tully"), ("Sansa Stark", "Roose Bolton")]:
    print(a, "->", b, shortest_path(a, b))

Daenerys Targaryen -> Red Wedding (4.1, ['Daenerys Targaryen', 'A Clash of Kings-Chapter 48', 'Red Wedding'])
Bran Stark -> Walder Frey (4.19, ['Bran Stark', 'A Storm of Swords-Chapter 65', 'Walder Frey'])
Jon Snow -> Edmure Tully (1.5, ['Jon Snow', 'Edmure Tully'])
Sansa Stark -> Roose Bolton (3.96, ['Sansa Stark', 'A Dance with Dragons-Chapter 12', 'Roose Bolton'])


In [148]:
nodes_out = [
    {"title": t, "kinds": corpus[t]["kinds"], "is_chapter": is_chapter(t),
     "degree": int(deg[t]), "pagerank_global": float(glob[idx[t]])}
    for t in nodes
]

In [149]:
edges_out = [
    {"a": a, "b": b, "kind": k, "cost": base_cost(a, b, k)}
    for (a, b), k in edges.items() if a < b
]

In [150]:
for name, rows in (("graph_nodes", nodes_out), ("graph_edges", edges_out)):
    with open(f"{CLEAN}/{name}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [151]:
print(len(nodes_out), len(edges_out))

3488 138721


In [152]:
page = corpus["A Clash of Kings-Chapter 48"]

In [153]:
print(page["text"][:150].replace("\n", " "))
print("Red Wedding" in page["text"], [h.group(2) for h in HEAD.finditer(page["text"])])
for t in ["A Storm of Swords-Chapter 51", "A Storm of Swords-Epilogue"]:
    print(t, "|", corpus[t]["text"][:120].replace("\n", " "))

Daenerys Targaryen enters the House of the Undying, where she experiences many visions. She reaches the audience chamber of the Undying Ones, who give
False ['Synopsis', 'Character List', 'Place and Terms Mentioned', 'Notes and References', 'External Links']
A Storm of Swords-Chapter 51 |  == Synopsis == Catelyn listens to the continuous pounding noise of the drums the musicians in the hall are playing. She
A Storm of Swords-Epilogue |  == Synopsis ==  Merrett Frey is approaching the ruins of Oldstones with the ransom for his half-brother's (Stevron Frey
